# CITE-seq BMMC GO importance and graph analysis

In [ ]:
mode = "all"


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

colors = [
    "#70acc0",
    "#c12075",
    "#003754",
    "#4c626c",
    "#be9e7c",
    "#723c5d",
    "#9d9c9c",
    "#00695c",
    "#e57373",
]
colors_name = [
    "graublau",
    "purpur",
    "blau",
    "dunkelgrau",
    "bronze",
    "violett",
    "hellgrau",
    "gedämpftes teal",
    "sanftes koralle",
]
sns.set_palette(sns.color_palette(colors))
base_color = "#70acc0"
light_palette = sns.light_palette(base_color, as_cmap=False)
colors = light_palette + ["#4c626c", "#003754"]
cmap = LinearSegmentedColormap.from_list("custom_cmap", colors)
colors_heat = ["#c12075", "#f0f0f0", "#003754"]
cmap_heat = LinearSegmentedColormap.from_list("custom_diverging", colors_heat, N=256)
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"

import re as _re
import matplotlib.figure as _mpl_figure

def _fix_svg_font_declarations(_svg_path):
    """Convert matplotlib's unsupported CSS 'font:' shorthand (labeled px, but
    actually point-sized, in a pt-calibrated viewBox) into explicit
    font-family/font-size/font-weight/font-style, which is what Figma's SVG
    importer requires to resolve size and style correctly."""
    try:
        with open(_svg_path, "r", encoding="utf-8") as _f:
            _content = _f.read()
    except OSError:
        return
    def _convert(_m):
        _oblique = _m.group(1)
        _weight = _m.group(2)
        _pt = float(_m.group(3)) * 0.75
        _family = _m.group(4)
        _font_weight = "bold" if _weight else "normal"
        _font_style = "oblique" if _oblique else "normal"
        return (
            f"font-family: {_family}; font-size: {_pt:g}pt; "
            f"font-weight: {_font_weight}; font-style: {_font_style}"
        )
    _pattern = _re.compile(r'font:\s*(oblique\s+)?(?:(\d+)\s+)?([0-9.]+)px\s+([^;"]+)')
    _new_content = _pattern.sub(_convert, _content)
    if _new_content != _content:
        with open(_svg_path, "w", encoding="utf-8") as _f:
            _f.write(_new_content)

_orig_figure_savefig = _mpl_figure.Figure.savefig

def _rasterize_dense_elements(_fig):
    for _ax in _fig.get_axes():
        for _coll in list(_ax.collections):
            try:
                _coll.set_rasterized(True)
            except Exception:
                pass
        for _im in _ax.get_images():
            try:
                _im.set_rasterized(True)
            except Exception:
                pass

def _patched_figure_savefig(self, fname, *args, **kwargs):
    if isinstance(fname, str) and fname.lower().endswith(".svg"):
        _rasterize_dense_elements(self)
        kwargs.setdefault("dpi", 300)
    _result = _orig_figure_savefig(self, fname, *args, **kwargs)
    if isinstance(fname, str) and fname.lower().endswith(".svg"):
        _fix_svg_font_declarations(fname)
    return _result

_mpl_figure.Figure.savefig = _patched_figure_savefig
mpl.rcParams["figure.dpi"] = 500.0
mpl.rcParams["savefig.transparent"] = True
mpl.rcParams["savefig.format"] = "png"

mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["DejaVu Sans"]

mpl.rcParams["axes.edgecolor"] = "black"
mpl.rcParams["axes.linewidth"] = 0.8
mpl.rcParams["axes.grid"] = True
mpl.rcParams["grid.color"] = "grey"
mpl.rcParams["grid.linestyle"] = "--"
mpl.rcParams["grid.linewidth"] = 0.5

mpl.rcParams["legend.frameon"] = False
mpl.rcParams["legend.loc"] = "best"

mpl.rcParams["xtick.major.size"] = 3
mpl.rcParams["xtick.minor.size"] = 2
mpl.rcParams["ytick.major.size"] = 3
mpl.rcParams["ytick.minor.size"] = 2
mpl.rcParams["xtick.major.width"] = 0.4
mpl.rcParams["xtick.minor.width"] = 0.3
mpl.rcParams["ytick.major.width"] = 0.4
mpl.rcParams["ytick.minor.width"] = 0.3
scaling_factor = 1
if mode == "all":
    base_font_size = 4
else:
    base_font_size = 6
base_axes_title_size = base_font_size + 1
base_axes_label_size = base_font_size
base_tick_label_size = base_font_size - 0.5
base_legend_font_size = base_font_size - 0.5
base_figure_title_size = base_font_size + 2
mpl.rcParams["font.size"] = base_font_size * scaling_factor
mpl.rcParams["axes.titlesize"] = base_axes_title_size * scaling_factor
mpl.rcParams["axes.labelsize"] = base_axes_label_size * scaling_factor
mpl.rcParams["xtick.labelsize"] = base_tick_label_size * scaling_factor
mpl.rcParams["ytick.labelsize"] = base_tick_label_size * scaling_factor
mpl.rcParams["legend.fontsize"] = base_legend_font_size * scaling_factor
mpl.rcParams["figure.titlesize"] = base_figure_title_size * scaling_factor

mpl.rcParams["patch.linewidth"] = 0.7

mpl.rcParams["figure.autolayout"] = True


In [ ]:
def set_figure_params(column, height):

    single_column_width = 3.5
    double_column_width = 7.0

    if column == "single":
        figure_width = single_column_width
    elif column == "double":
        figure_width = double_column_width
    else:
        figure_width = column
    scaling_factor = figure_width / single_column_width
    scaling_factor = 1

    mpl.rcParams["figure.figsize"] = (figure_width, height)


In [ ]:
import sys
import os

_OUTDIR_ABS = os.path.join(os.getcwd(), "outputs", "fig2dg_S16_S17_neurips2021_cite_bmmc_go_graph")
os.makedirs(_OUTDIR_ABS, exist_ok=True)

sys.path.append(os.path.abspath("../.experiment_data/primary/benchmarking"))

import contextlib
import functools

_POSTPROC_CWD = os.getcwd()

@contextlib.contextmanager
def _in_primary_dir():
    os.chdir(os.path.join(_POSTPROC_CWD, "../.experiment_data/primary"))
    try:
        yield
    finally:
        os.chdir(_POSTPROC_CWD)

with _in_primary_dir():
    import muon as _muon
    _orig_sample_obs = _muon.pp.sample_obs
    def _patched_sample_obs(data, *a, **kw):
        return _orig_sample_obs(data, *a, **kw).copy()
    _muon.pp.sample_obs = _patched_sample_obs
    import datasets

for _name in dir(datasets):
    if _name.startswith("load_"):
        def _make_wrapper(fn):
            @functools.wraps(fn)
            def _wrapper(*a, **kw):
                with _in_primary_dir():
                    return fn(*a, **kw)
            return _wrapper
        setattr(datasets, _name, _make_wrapper(getattr(datasets, _name)))
import glob
import pandas as pd
import scipy
import numpy as np
import os
from tqdm import tqdm
import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr
from scipy.spatial.distance import euclidean
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pickle
import glob
from collections import Counter
from pathlib import Path
from goatools.obo_parser import GODag
from goatools.anno.gaf_reader import GafReader
import glob
import pandas as pd
import scipy
import numpy as np
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap
import textwrap
from matplotlib.colors import TwoSlopeNorm


In [ ]:
obodag = GODag(
    os.path.join(
        os.path.abspath("../.experiment_data/primary"), "preprocessing", "resources", "ebi_ensembl_go_mappings", "go-basic.obo"
    )
)


In [ ]:
if mode == "all":
    phenotypes = [
        "HSC",
        "Lymph_prog",
        "G_M_prog",
        "MK_E_prog",
        "ILC",
        "ILC1",
        "NK",
        "NK_CD158e1+",
        "CD4+_T_naive",
        "CD4+_T_activated",
        "CD4+_T_activated_integrinB7+",
        "CD4+_T_CD314+_CD45RA+",
        "CD8+_T_naive",
        "CD8+_T_naive_CD127+_CD26-_CD101-",
        "T_prog_cycling",
        "CD8+_T_CD49f+",
        "CD8+_T_CD69+_CD45RA+",
        "CD8+_T_CD69+_CD45RO+",
        "CD8+_T_TIGIT+_CD45RA+",
        "CD8+_T_TIGIT+_CD45RO+",
        "CD8+_T_CD57+_CD45RA+",
        "CD8+_T_CD57+_CD45RO+",
        "gdT_CD158b+",
        "gdT_TCRVD2+",
        "dnT",
        "T_reg",
        "Naive_CD20+_B_IGKC+",
        "Naive_CD20+_B_IGKC-",
        "Transitional_B",
        "B1_B_IGKC+",
        "B1_B_IGKC-",
        "Plasma_cell_IGKC+",
        "Plasma_cell_IGKC-",
        "Plasmablast_IGKC+",
        "Plasmablast_IGKC-",
        "cDC1",
        "cDC2",
        "pDC",
        "CD14+_Mono",
        "CD16+_Mono",
        "Erythroblast",
        "Normoblast",
        "Proerythroblast",
        "Reticulocyte",
        "MAIT",
    ]
    phenotypes_labels = [
        "HSC",
        "Lymph prog",
        "G/M prog",
        "MK/E prog",
        "ILC",
        "ILC1",
        "NK",
        "NK CD158e1+",
        "CD4+ T naive",
        "CD4+ T activated",
        "CD4+ T activated B7+",
        "CD4+ T CD314+ CD45RA+",
        "CD8+ T naive",
        "CD8+ T naive CD127+ CD26- CD101-",
        "T prog cycling",
        "CD8+ T CD49f+",
        "CD8+ T CD69+ CD45RA+",
        "CD8+ T CD69+ CD45RO+",
        "CD8+ T TIGIT+ CD45RA+",
        "CD8+ T TIGIT+ CD45RO+",
        "CD8+ T CD57+ CD45RA+",
        "CD8+ T CD57+ CD45RO+",
        "gdT CD158b+",
        "gdT TCRVD2+",
        "dnT",
        "T reg",
        "Naive CD20+ B IGKC+",
        "Naive CD20+ B IGKC-",
        "Transitional B",
        "B1 B IGKC+",
        "B1 B IGKC-",
        "Plasma cell IGKC+",
        "Plasma cell IGKC-",
        "Plasmablast IGKC+",
        "Plasmablast IGKC-",
        "cDC1",
        "cDC2",
        "pDC",
        "CD14+ Mono",
        "CD16+ Mono",
        "Erythroblast",
        "Normoblast",
        "Proerythroblast",
        "Reticulocyte",
        "MAIT",
    ]
    phenotypes_dataset = phenotypes_labels
else:
    phenotypes = [
        "Lymph_prog",
        "CD4+_T_naive",
        "CD4+_T_activated",
        "CD4+_T_activated_integrinB7+",
        "CD4+_T_CD314+_CD45RA+",
        "CD8+_T_naive",
        "CD8+_T_naive_CD127+_CD26-_CD101-",
        "CD8+_T_TIGIT+_CD45RA+",
        "CD8+_T_TIGIT+_CD45RO+",
        "CD8+_T_CD57+_CD45RA+",
        "CD8+_T_CD57+_CD45RO+",
        "Naive_CD20+_B_IGKC+",
        "Naive_CD20+_B_IGKC-",
        "Erythroblast",
        "Normoblast",
        "Reticulocyte",
    ]
    phenotypes_labels = [
        "Lymph prog",
        "CD4+ T naive",
        "CD4+ T activated",
        "CD4+ T activated integrinB7+",
        "CD4+ T CD314+ CD45RA+",
        "CD8+ T naive",
        "CD8+ T naive CD127+ CD26- CD101-",
        "CD8+ T TIGIT+ CD45RA+",
        "CD8+ T TIGIT+ CD45RO+",
        "CD8+ T CD57+ CD45RA+",
        "CD8+ T CD57+ CD45RO+",
        "Naive CD20+ B IGKC+",
        "Naive CD20+ B IGKC-",
        "Erythroblast",
        "Normoblast",
        "Reticulocyte",
    ]
    phenotypes_dataset = phenotypes_labels
    phenotypes = [
        "Lymph_prog",
        "G_M_prog",
        "MK_E_prog",
        "NK",
        "CD4+_T_naive",
        "CD4+_T_activated",
        "CD4+_T_activated_integrinB7+",
        "CD4+_T_CD314+_CD45RA+",
        "CD8+_T_naive",
        "CD8+_T_naive_CD127+_CD26-_CD101-",
        "CD8+_T_TIGIT+_CD45RA+",
        "CD8+_T_TIGIT+_CD45RO+",
        "CD8+_T_CD57+_CD45RA+",
        "CD8+_T_CD57+_CD45RO+",
        "Naive_CD20+_B_IGKC+",
        "Naive_CD20+_B_IGKC-",
        "Plasma_cell_IGKC+",
        "Plasma_cell_IGKC-",
        "Plasmablast_IGKC+",
        "Plasmablast_IGKC-",
        "cDC1",
        "cDC2",
        "CD14+_Mono",
        "CD16+_Mono",
        "Erythroblast",
        "Normoblast",
        "Reticulocyte",
    ]
    phenotypes_labels = [
        "Lymph prog",
        "G/M prog",
        "MK/E prog",
        "NK",
        "CD4+ T naive",
        "CD4+ T activated",
        "CD4+ T activated integrinB7+",
        "CD4+ T CD314+ CD45RA+",
        "CD8+ T naive",
        "CD8+ T naive CD127+ CD26- CD101-",
        "CD8+ T TIGIT+ CD45RA+",
        "CD8+ T TIGIT+ CD45RO+",
        "CD8+ T CD57+ CD45RA+",
        "CD8+ T CD57+ CD45RO+",
        "Naive CD20+ B IGKC+",
        "Naive CD20+ B IGKC-",
        "Plasma cell IGKC+",
        "Plasma cell IGKC-",
        "Plasmablast IGKC+",
        "Plasmablast IGKC-",
        "cDC1",
        "cDC2",
        "CD14+ Mono",
        "CD16+ Mono",
        "Erythroblast",
        "Normoblast",
        "Reticulocyte",
    ]
    phenotypes_dataset = phenotypes_labels


In [ ]:
phenotypes_cd8_tcells = [
    "CD8+_T_naive",
    "CD8+_T_naive_CD127+_CD26-_CD101-",
    "CD8+_T_CD49f+",
    "CD8+_T_CD69+_CD45RA+",
    "CD8+_T_CD69+_CD45RO+",
    "CD8+_T_TIGIT+_CD45RA+",
    "CD8+_T_TIGIT+_CD45RO+",
    "CD8+_T_CD57+_CD45RA+",
    "CD8+_T_CD57+_CD45RO+",
]
phenotypes_cd8_tcells_labels = [
    "CD8+ T naive",
    "CD8+ T naive CD127+ CD26- CD101-",
    "CD8+ T CD49f+",
    "CD8+ T CD69+ CD45RA+",
    "CD8+ T CD69+ CD45RO+",
    "CD8+ T TIGIT+ CD45RA+",
    "CD8+ T TIGIT+ CD45RO+",
    "CD8+ T CD57+ CD45RA+",
    "CD8+ T CD57+ CD45RO+",
]
phenotypes_cd8_tcells_dataset = phenotypes_cd8_tcells_labels


In [ ]:
phenotypes_cd8_tcells = [
    "CD8+_T_naive",
    "CD8+_T_naive_CD127+_CD26-_CD101-",
    "CD8+_T_TIGIT+_CD45RA+",
    "CD8+_T_TIGIT+_CD45RO+",
    "CD8+_T_CD57+_CD45RA+",
    "CD8+_T_CD57+_CD45RO+",
]
phenotypes_cd8_tcells_labels = [
    "CD8+ T naive",
    "CD8+ T naive CD127+ CD26- CD101-",
    "CD8+ T TIGIT+ CD45RA+",
    "CD8+ T TIGIT+ CD45RO+",
    "CD8+ T CD57+ CD45RA+",
    "CD8+ T CD57+ CD45RO+",
]
phenotypes_cd8_tcells_dataset = phenotypes_cd8_tcells_labels


In [ ]:
(
    mdata,
    mdata_counts,
    adata,
    train_indices,
    val_indices,
    n_genes,
    n_regions,
    n_snps,
    labels_key,
    further_labels_keys,
    batch_key,
    patient_key,
    further_batch_keys,
    further_continuous_batch_keys,
    n_patient_covariates,
    protein_expression_obsm_key,
) = datasets.load_neurips2021_cite_BMMC(1.0, False, True)


In [ ]:
def aggregate_dataframes(graph_csv_list):
    num_rows = len(graph_csv_list[0])
    for df in graph_csv_list:
        if len(df) != num_rows:
            raise ValueError("Alle DataFrames müssen die gleiche Anzahl an Zeilen haben.")

    aggregated_roc = []
    aggregated_activations = []

    for row_idx in range(num_rows):

        roc_values = [df.iloc[row_idx]["predictors_label_valid_roc"] for df in graph_csv_list]

        mean_roc = np.mean(roc_values, axis=0)
        aggregated_roc.append(mean_roc)

        activations_arrays = [df.iloc[row_idx]["activations_valid"] for df in graph_csv_list]

        concatenated_activations = np.concatenate(activations_arrays, axis=1)

        mean_activations = np.mean(concatenated_activations, axis=1)
        aggregated_activations.append(mean_activations)

    aggregated_df = pd.DataFrame(
        index=graph_csv_list[0].index,
        data={
            "predictors_label_valid_roc": aggregated_roc,
            "activations_valid": aggregated_activations,
        },
    )

    return aggregated_df


In [ ]:
seeds = ["0"]

graph_csv_list = []
graph_csv_index_list = []
go_importances = []
for modality in tqdm(["expression", "protein"]):
    go_importances_modality = []
    graph_csv_modality_list = []
    for (
        phenotype
    ) in phenotypes:
        graph_csv_phenotypes_list = []
        graph_csv = pd.read_pickle(
            f"../.experiment_data/scratch/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/calc_go_terms_cell_type/{modality}_cell_type_vs_rest_{phenotype}_res_unp_act_mask_False_flip_False_go_terms_evaluated_graph_csv.pkl"
        )
        graph_csv_phenotypes_list.append(graph_csv)
        graph_csv = aggregate_dataframes(graph_csv_phenotypes_list)
        graph_csv_modality_list.append(graph_csv)
        go_importances_modality.append(np.array(graph_csv["predictors_label_valid_roc"]))
    graph_csv_list.append(graph_csv_modality_list)
    graph_csv_index_list.append(np.array(graph_csv_modality_list[0].index))
    go_importances.append(go_importances_modality)


In [ ]:
for go_term in ["GO:0032728", "GO:0042110", "GO:0007155", "GO:0002921"]:
    print(go_term in graph_csv_list[0][0].index)


In [ ]:
for go_term in ["GO:0002478", "GO:0019882", "GO:0006886", "GO:0007259", "GO:0034340", "GO:0006955"]:
    print(go_term in graph_csv_list[0][0].index)


In [ ]:
for go_term in [
    "GO:0006270",
    "GO:0006399",
    "GO:0006783",
    "GO:0034728",
    "GO:0006351",
    "GO:0048511",
    "GO:0016236",
    "GO:0098772",
    "GO:0046488",
    "GO:0048471",
]:
    print(go_term in graph_csv_list[0][0].index)


In [ ]:
seeds = ["0"]

graph_csv_dense_list = []
graph_csv_dense_index_list = []
go_dense_importances = []
for modality in tqdm(["expression", "protein"]):
    go_dense_importances_modality = []
    graph_csv_dense_modality_list = []
    for (
        phenotype
    ) in phenotypes:
        graph_csv_dense_phenotypes_list = []
        graph_csv = pd.read_pickle(
            f"../.experiment_data/scratch/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_dense_s0/default/calc_go_terms_cell_type/{modality}_cell_type_vs_rest_{phenotype}_res_unp_act_mask_False_flip_False_go_terms_evaluated_graph_csv.pkl"
        )
        graph_csv_dense_phenotypes_list.append(graph_csv)
        graph_csv = aggregate_dataframes(graph_csv_dense_phenotypes_list)
        graph_csv_dense_modality_list.append(graph_csv)
        go_dense_importances_modality.append(np.array(graph_csv["predictors_label_valid_roc"]))
    graph_csv_dense_list.append(graph_csv_dense_modality_list)
    graph_csv_dense_index_list.append(np.array(graph_csv_dense_modality_list[0].index))
    go_dense_importances.append(go_dense_importances_modality)


In [ ]:
seeds = ["0"]

graph_csv_cd8_tcells_list = []
graph_csv_cd8_tcells_index_list = []
go_cd8_tcells_importances = []
for modality in tqdm(["expression", "protein"]):
    go_cd8_tcells_importances_modality = []
    graph_csv_cd8_tcells_modality_list = []
    for (
        phenotype
    ) in phenotypes_cd8_tcells:
        graph_csv_cd8_tcells_phenotypes_list = []
        graph_csv = pd.read_pickle(
            f"../.experiment_data/scratch/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/calc_go_terms_cell_type/{modality}_cell_type_vs_rest_{phenotype}_res_unp_act_mask_False_flip_False_res_cell_type_CD8+ T cell_go_terms_evaluated_graph_csv.pkl"
        )
        graph_csv_cd8_tcells_phenotypes_list.append(graph_csv)
        graph_csv = aggregate_dataframes(graph_csv_cd8_tcells_phenotypes_list)
        graph_csv_cd8_tcells_modality_list.append(graph_csv)
        go_cd8_tcells_importances_modality.append(np.array(graph_csv["predictors_label_valid_roc"]))
    graph_csv_cd8_tcells_list.append(graph_csv_cd8_tcells_modality_list)
    graph_csv_cd8_tcells_index_list.append(np.array(graph_csv_cd8_tcells_modality_list[0].index))
    go_cd8_tcells_importances.append(go_cd8_tcells_importances_modality)


In [ ]:
np.unique([arr.shape for arr in graph_csv["activations_valid"]])


In [ ]:
indices_cd8_tcells = np.where(
    mdata_counts.mod["rna"]
    .obs[labels_key][train_indices]
    .isin(
        [
            "HSC",
            "CD8+ T naive",
            "CD8+ T naive CD127+ CD26- CD101-",
            "CD8+ T CD49f+",
            "CD8+ T CD69+ CD45RA+",
            "CD8+ T CD69+ CD45RO+",
            "CD8+ T TIGIT+ CD45RA+",
            "CD8+ T TIGIT+ CD45RO+",
            "CD8+ T CD57+ CD45RA+",
            "CD8+ T CD57+ CD45RO+",
        ]
    )
)[0]


In [ ]:
go_activation_space_sparsevi_expression = np.concatenate(
    pd.read_pickle(
        f"../.experiment_data/scratch/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/calc_go_terms_cell_type/expression_cell_type_vs_rest_{phenotypes_cd8_tcells[0]}_res_unp_act_mask_False_flip_False_res_cell_type_CD8+ T cell_go_terms_evaluated_graph_csv.pkl"
    )["activations_valid"].values.tolist(),
    axis=1,
)


In [ ]:
go_activation_space_sparsevi_protein = np.concatenate(
    pd.read_pickle(
        f"../.experiment_data/scratch/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/calc_go_terms_cell_type/protein_cell_type_vs_rest_{phenotypes_cd8_tcells[0]}_res_unp_act_mask_False_flip_False_res_cell_type_CD8+ T cell_go_terms_evaluated_graph_csv.pkl"
    )["activations_valid"].values.tolist(),
    axis=1,
)


In [ ]:
go_activation_space_sparsevi = np.concatenate(
    [go_activation_space_sparsevi_expression, go_activation_space_sparsevi_protein], axis=1
)


In [ ]:
latent_sparsevi = np.load(
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/X_sparsevi.npy"
)
latent_sparsevi = latent_sparsevi[indices_cd8_tcells]
latent_sparsevi


In [ ]:
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    homogeneity_completeness_v_measure,
    fowlkes_mallows_score,
)
import matplotlib.pyplot as plt

data = np.array([[1, 2], [2, 3], [3, 4], [5, 6], [8, 9]])
true_labels = [1, 1, 1, 2, 2]
data = go_activation_space_sparsevi
true_labels = mdata_counts.mod["rna"].obs[labels_key][train_indices][indices_cd8_tcells]


In [ ]:
data = go_activation_space_sparsevi_expression


In [ ]:
data = go_activation_space_sparsevi_protein


In [ ]:
data = go_activation_space_sparsevi


In [ ]:
data = latent_sparsevi


# GO importance statistics

In [ ]:
for modality_index, modality in tqdm(enumerate(['expression', 'protein'])):
    set_figure_params('double', 9)
    fig, axes = plt.subplots(9, 5)
    axes = axes.flatten()
    for i, (ax, _) in enumerate(zip(axes, go_importances[modality_index])):
        print(phenotypes_labels[i])
        print(np.mean(go_importances[modality_index][i]))
        ax.hist(go_importances[modality_index][i], bins=50)
        ax.set_title(phenotypes_labels[i])
        ax.set_xlabel('GO importance')
        ax.set_ylabel('prevalence')
        ax.set_xlim(right=1.0)
    plt.tight_layout()
    plt.show()


In [ ]:
for modality_index, modality in tqdm(enumerate(['expression', 'protein'])):
    set_figure_params('double', 9)
    fig, axes = plt.subplots(9, 5)
    axes = axes.flatten()
    for i, (ax, _) in enumerate(zip(axes, go_dense_importances[modality_index])):
        print(phenotypes_labels[i])
        print(np.mean(go_dense_importances[modality_index][i]))
        ax.hist(go_dense_importances[modality_index][i], bins=50)
        ax.set_title(phenotypes_labels[i])
        ax.set_xlabel('GO importance')
        ax.set_ylabel('prevalence')
        ax.set_xlim(right=1.0)
    plt.tight_layout()
    plt.show()


In [ ]:
for modality_index, modality in tqdm(enumerate(['expression', 'protein'])):
    set_figure_params(7.0 / 5 * 3, 9 / 9 * 2)
    fig, axes = plt.subplots(2, 3)
    axes = axes.flatten()
    for i, (ax, _) in enumerate(zip(axes, go_cd8_tcells_importances[modality_index])):
        ax.hist(go_cd8_tcells_importances[modality_index][i], bins=50)
        ax.set_title(phenotypes_cd8_tcells_labels[i])
        ax.set_xlabel('GO importance')
        ax.set_ylabel('prevalence')
        ax.set_xlim(right=1.0)
    plt.tight_layout()
    plt.show()
    plt.savefig(os.path.join(_OUTDIR_ABS, f'neurips2021_cite_bmmc_go_importances_hist_cd8_{modality}_{mode}.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
    plt.savefig(os.path.join(_OUTDIR_ABS, f'neurips2021_cite_bmmc_go_importances_hist_cd8_{modality}_{mode}.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')


# GO Graph Similarity

In [ ]:
def calculate_distance_matrix(data, metric='spearman'):
    n = len(data)
    distance_matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(i, n):
            if metric == 'spearman':
                distance, _ = scipy.stats.spearmanr(data[i], data[j])
            elif metric == 'pearson':
                distance, _ = scipy.stats.pearsonr(data[i], data[j])
            elif metric == 'euclidean':
                distance = scipy.spatial.distance.euclidean(data[i], data[j])
            else:
                raise ValueError("Unknown metric. Use 'spearman', 'pearson', or 'euclidean'.")
            distance_matrix[i, j] = distance
            distance_matrix[j, i] = distance
    return distance_matrix

def calculate_distance_matrix_combinations(data1, data2, metric='spearman'):
    n = len(data1)
    distance_matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(i, n):
            if metric == 'spearman':
                distance, _ = scipy.stats.spearmanr(data1[i], data2[j])
            elif metric == 'pearson':
                distance, _ = scipy.stats.pearsonr(data1[i], data2[j])
            elif metric == 'euclidean':
                distance = scipy.spatial.distance.euclidean(data1[i], data2[j])
            else:
                raise ValueError("Unknown metric. Use 'spearman', 'pearson', or 'euclidean'.")
            distance_matrix[i, j] = distance
            distance_matrix[j, i] = distance
    return distance_matrix

def plot_heatmap_go_graph_similarity(distance_matrix, distance_matrix_mask, xlabels, ylabels, part_indices, vmin=0, figsize=None, metric='Metric', mirrored_top=False, mirrored_right=False, save_name=False, format='png'):
    for n_part_index, part_index in enumerate(part_indices):
        distance_matrix = np.insert(distance_matrix, part_index + n_part_index, np.zeros((1, distance_matrix.shape[0])), axis=0)
        distance_matrix = np.insert(distance_matrix, part_index + n_part_index, np.zeros((1, distance_matrix.shape[0])), axis=1)
        distance_matrix_mask = np.insert(distance_matrix_mask, part_index + n_part_index, np.ones((1, distance_matrix_mask.shape[0])), axis=0)
        distance_matrix_mask = np.insert(distance_matrix_mask, part_index + n_part_index, np.ones((1, distance_matrix_mask.shape[0])), axis=1)
    if xlabels is not None and ylabels is not None:
        for part_index in sorted(part_indices, reverse=True):
            xlabels.insert(part_index, '')
            ylabels.insert(part_index, '')
    if figsize is not None:
        set_figure_params(figsize[0], figsize[1])
    else:
        set_figure_params('double', 5.8)
    fig, ax = plt.subplots()
    if mirrored_right:
        cbar = False
    else:
        cbar = True
    if xlabels is not None and ylabels is not None:
        if metric == 'spearman' or metric == 'pearson':
            midpoint = 0
            vmax = 1
            norm = TwoSlopeNorm(vmin=vmin, vcenter=midpoint, vmax=vmax)
            sns.heatmap(distance_matrix, norm=norm, mask=distance_matrix_mask, xticklabels=xlabels, yticklabels=ylabels, cmap=cmap_heat, ax=ax, cbar=cbar, cbar_kws={'fraction': 0.01, 'pad': 0.01})
        else:
            sns.heatmap(distance_matrix, mask=distance_matrix_mask, xticklabels=xlabels, yticklabels=ylabels, cmap=cmap_heat, ax=ax, cbar=cbar, cbar_kws={'fraction': 0.01, 'pad': 0.01})
    elif metric == 'spearman' or metric == 'pearson':
        midpoint = 0
        vmax = 1
        norm = TwoSlopeNorm(vmin=vmin, vcenter=midpoint, vmax=vmax)
        sns.heatmap(distance_matrix, norm=norm, mask=distance_matrix_mask, xticklabels=False, yticklabels=False, cmap=cmap_heat, ax=ax, cbar=cbar, cbar_kws={'fraction': 0.01, 'pad': 0.01})
    else:
        sns.heatmap(distance_matrix, mask=distance_matrix_mask, xticklabels=False, yticklabels=False, cmap=cmap_heat, ax=ax, cbar=cbar, cbar_kws={'fraction': 0.01, 'pad': 0.01})
    ax.grid(False)
    n_rows, n_cols = distance_matrix_mask.shape
    for i in range(n_rows):
        for j in range(n_cols):
            if not distance_matrix_mask[i, j]:
                ax.hlines(i + 0.5, j, j + 1, color='black', linewidth=0.2)
                ax.vlines(j + 0.5, i, i + 1, color='black', linewidth=0.2)
    if mirrored_top:
        ax.xaxis.set_ticks_position('top')
        ax.xaxis.tick_top()
        ax.yaxis.set_ticks_position('left')
        ax.yaxis.tick_left()
        ax.tick_params(axis='x', rotation=90)
        ax.tick_params(axis='y', rotation=0)
    if mirrored_right:
        ax.xaxis.set_ticks_position('bottom')
        ax.xaxis.tick_bottom()
        ax.yaxis.set_ticks_position('right')
        ax.yaxis.tick_right()
        ax.tick_params(axis='x', rotation=90)
        ax.tick_params(axis='y', rotation=0)
    ax.tick_params(axis='x')
    ax.tick_params(axis='y')
    if xlabels is not None and ylabels is not None:
        for i, label in enumerate(xlabels):
            if label == '':
                ax.xaxis.get_major_ticks()[i].tick1line.set_visible(False)
                ax.xaxis.get_major_ticks()[i].tick2line.set_visible(False)
        for i, label in enumerate(ylabels):
            if label == '':
                ax.yaxis.get_major_ticks()[i].tick1line.set_visible(False)
                ax.yaxis.get_major_ticks()[i].tick2line.set_visible(False)
    else:
        ax.tick_params(axis='x', length=0)
        ax.tick_params(axis='y', length=0)
    ax.set_rasterized(True)
    plt.tight_layout()
    if save_name:
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')
    else:
        plt.show()


In [ ]:
vmin_list = [
    np.concatenate(
        [
            calculate_distance_matrix(go_importances[0], metric=metric).flatten(),
            calculate_distance_matrix(go_importances[1], metric=metric).flatten(),
            calculate_distance_matrix(go_cd8_tcells_importances[0], metric=metric).flatten(),
            calculate_distance_matrix(go_cd8_tcells_importances[1], metric=metric).flatten(),
        ]
    ).min()
    for metric in ["spearman"]
]
vmin_list


In [ ]:
for metric, vmin in tqdm(zip(["spearman"], vmin_list)):
    distance_matrix = np.zeros((2 * len(phenotypes), 2 * len(phenotypes)))
    distance_matrix_mask = np.zeros((2 * len(phenotypes), 2 * len(phenotypes)))
    distance_matrix_mask[: len(phenotypes), len(phenotypes) : 2 * len(phenotypes)] = 1

    distance_matrix[: len(phenotypes), : len(phenotypes)] = calculate_distance_matrix(
        go_importances[0], metric=metric
    )
    distance_matrix_mask[: len(phenotypes), : len(phenotypes)] = np.triu(
        np.ones_like(distance_matrix[: len(phenotypes), : len(phenotypes)], dtype=bool)
    )
    distance_matrix[
        len(phenotypes) : 2 * len(phenotypes), len(phenotypes) : 2 * len(phenotypes)
    ] = calculate_distance_matrix(go_importances[1], metric=metric)
    distance_matrix_mask[
        len(phenotypes) : 2 * len(phenotypes), len(phenotypes) : 2 * len(phenotypes)
    ] = np.triu(np.ones_like(distance_matrix[: len(phenotypes), : len(phenotypes)], dtype=bool))

    go_importances_0_filt = np.array(go_importances[0])[
        :,
        np.isin(
            graph_csv_index_list[0],
            np.intersect1d(graph_csv_index_list[0], graph_csv_index_list[1]),
        ),
    ]
    go_importances_1_filt = np.array(go_importances[1])[
        :,
        np.isin(
            graph_csv_index_list[1],
            np.intersect1d(graph_csv_index_list[1], graph_csv_index_list[0]),
        ),
    ]
    distance_matrix[len(phenotypes) : 2 * len(phenotypes), : len(phenotypes)] = (
        calculate_distance_matrix_combinations(
            go_importances_0_filt, go_importances_1_filt, metric=metric
        )
    )
    distance_matrix_mask[len(phenotypes) : 2 * len(phenotypes), : len(phenotypes)] = np.triu(
        np.ones_like(distance_matrix[: len(phenotypes), : len(phenotypes)], dtype=bool), k=1
    )

    if mode == "all":
        plot_heatmap_go_graph_similarity(
            distance_matrix,
            distance_matrix_mask,
            phenotypes_labels + phenotypes_labels[:-1] + [""],
            [""] + phenotypes_labels[1:] + phenotypes_labels,
            [len(phenotypes), len(phenotypes)],
            vmin=vmin,
            figsize=(7, 5.8),
            metric=metric,
            save_name=f"neurips2021_cite_bmmc_go_graph_{mode}_{metric}",
        )
    else:
        plot_heatmap_go_graph_similarity(
            distance_matrix,
            distance_matrix_mask,
            phenotypes_labels + phenotypes_labels[:-1] + [""],
            [""] + phenotypes_labels[1:] + phenotypes_labels,
            [len(phenotypes)],
            vmin=vmin,
            figsize=(7, 6.4),
            metric=metric,
            save_name=f"neurips2021_cite_bmmc_go_graph_{mode}_{metric}",
        )


In [ ]:
for metric, vmin in tqdm(zip(["spearman"], vmin_list)):
    distance_matrix = np.zeros((2 * len(phenotypes), 2 * len(phenotypes)))
    distance_matrix_mask = np.zeros((2 * len(phenotypes), 2 * len(phenotypes)))
    distance_matrix_mask[: len(phenotypes), len(phenotypes) : 2 * len(phenotypes)] = 1

    distance_matrix[: len(phenotypes), : len(phenotypes)] = calculate_distance_matrix(
        go_importances[0], metric=metric
    )
    distance_matrix_mask[: len(phenotypes), : len(phenotypes)] = np.triu(
        np.ones_like(distance_matrix[: len(phenotypes), : len(phenotypes)], dtype=bool)
    )
    distance_matrix[
        len(phenotypes) : 2 * len(phenotypes), len(phenotypes) : 2 * len(phenotypes)
    ] = calculate_distance_matrix(go_importances[1], metric=metric)
    distance_matrix_mask[
        len(phenotypes) : 2 * len(phenotypes), len(phenotypes) : 2 * len(phenotypes)
    ] = np.triu(np.ones_like(distance_matrix[: len(phenotypes), : len(phenotypes)], dtype=bool))

    go_importances_0_filt = np.array(go_importances[0])[
        :,
        np.isin(
            graph_csv_index_list[0],
            np.intersect1d(graph_csv_index_list[0], graph_csv_index_list[1]),
        ),
    ]
    go_importances_1_filt = np.array(go_importances[1])[
        :,
        np.isin(
            graph_csv_index_list[1],
            np.intersect1d(graph_csv_index_list[1], graph_csv_index_list[0]),
        ),
    ]
    distance_matrix[len(phenotypes) : 2 * len(phenotypes), : len(phenotypes)] = (
        calculate_distance_matrix_combinations(
            go_importances_0_filt, go_importances_1_filt, metric=metric
        )
    )
    distance_matrix_mask[len(phenotypes) : 2 * len(phenotypes), : len(phenotypes)] = np.triu(
        np.ones_like(distance_matrix[: len(phenotypes), : len(phenotypes)], dtype=bool), k=1
    )

    if mode == "all":
        plot_heatmap_go_graph_similarity(
            np.rot90(distance_matrix, k=3),
            np.rot90(distance_matrix_mask, k=3),
            phenotypes_labels[::-1] + phenotypes_labels[:-1][::-1] + [""],
            phenotypes_labels + phenotypes_labels[:-1] + [""],
            [len(phenotypes), len(phenotypes)],
            vmin=vmin,
            figsize=(7, 5.8),
            metric=metric,
            mirrored_top=True,
            save_name=f"neurips2021_cite_bmmc_go_graph_rotated_{mode}_{metric}",
        )
    else:
        plot_heatmap_go_graph_similarity(
            np.rot90(distance_matrix, k=3),
            np.rot90(distance_matrix_mask, k=3),
            phenotypes_labels[::-1] + phenotypes_labels[:-1][::-1] + [""],
            phenotypes_labels + phenotypes_labels[:-1] + [""],
            [len(phenotypes)],
            vmin=vmin,
            figsize=(7, 6.4),
            metric=metric,
            mirrored_top=True,
            save_name=f"neurips2021_cite_bmmc_go_graph_rotated_{mode}_{metric}",
        )


In [ ]:
for metric, vmin in tqdm(zip(["spearman"], vmin_list)):
    distance_matrix = np.zeros((2 * len(phenotypes), 2 * len(phenotypes)))
    distance_matrix_mask = np.zeros((2 * len(phenotypes), 2 * len(phenotypes)))
    distance_matrix_mask[: len(phenotypes), len(phenotypes) : 2 * len(phenotypes)] = 1

    distance_matrix[: len(phenotypes), : len(phenotypes)] = calculate_distance_matrix(
        go_importances[0], metric=metric
    )
    distance_matrix_mask[: len(phenotypes), : len(phenotypes)] = np.triu(
        np.ones_like(distance_matrix[: len(phenotypes), : len(phenotypes)], dtype=bool)
    )
    distance_matrix[
        len(phenotypes) : 2 * len(phenotypes), len(phenotypes) : 2 * len(phenotypes)
    ] = calculate_distance_matrix(go_importances[1], metric=metric)
    distance_matrix_mask[
        len(phenotypes) : 2 * len(phenotypes), len(phenotypes) : 2 * len(phenotypes)
    ] = np.triu(np.ones_like(distance_matrix[: len(phenotypes), : len(phenotypes)], dtype=bool))

    go_importances_0_filt = np.array(go_importances[0])[
        :,
        np.isin(
            graph_csv_index_list[0],
            np.intersect1d(graph_csv_index_list[0], graph_csv_index_list[1]),
        ),
    ]
    go_importances_1_filt = np.array(go_importances[1])[
        :,
        np.isin(
            graph_csv_index_list[1],
            np.intersect1d(graph_csv_index_list[1], graph_csv_index_list[0]),
        ),
    ]
    distance_matrix[len(phenotypes) : 2 * len(phenotypes), : len(phenotypes)] = (
        calculate_distance_matrix_combinations(
            go_importances_0_filt, go_importances_1_filt, metric=metric
        )
    )
    distance_matrix_mask[len(phenotypes) : 2 * len(phenotypes), : len(phenotypes)] = np.triu(
        np.ones_like(distance_matrix[: len(phenotypes), : len(phenotypes)], dtype=bool), k=1
    )

    if mode == "all":
        plot_heatmap_go_graph_similarity(
            np.rot90(distance_matrix, k=3),
            np.rot90(distance_matrix_mask, k=3),
            phenotypes_labels[::-1] + phenotypes_labels[:-1][::-1] + [""],
            phenotypes_labels + phenotypes_labels[:-1] + [""],
            [len(phenotypes), len(phenotypes)],
            vmin=vmin,
            figsize=(7, 5.8),
            metric=metric,
            mirrored_top=True,
            save_name=f"neurips2021_cite_bmmc_go_graph_rotated_{mode}_{metric}",
        )
    else:
        plot_heatmap_go_graph_similarity(
            np.rot90(distance_matrix, k=3),
            np.rot90(distance_matrix_mask, k=3),
            phenotypes_labels[::-1] + phenotypes_labels[:-1][::-1] + [""],
            phenotypes_labels + phenotypes_labels[:-1] + [""],
            [len(phenotypes)],
            vmin=vmin,
            figsize=(7, 6.4),
            metric=metric,
            mirrored_top=True,
            save_name=f"neurips2021_cite_bmmc_go_graph_rotated_{mode}_{metric}",
        )


In [ ]:
for metric, vmin in tqdm(zip(["spearman"], vmin_list)):
    distance_matrix = np.zeros((2 * len(phenotypes), 2 * len(phenotypes)))
    distance_matrix_mask = np.zeros((2 * len(phenotypes), 2 * len(phenotypes)))
    distance_matrix_mask[: len(phenotypes), len(phenotypes) : 2 * len(phenotypes)] = 1

    distance_matrix[: len(phenotypes), : len(phenotypes)] = calculate_distance_matrix(
        go_dense_importances[0], metric=metric
    )
    distance_matrix_mask[: len(phenotypes), : len(phenotypes)] = np.triu(
        np.ones_like(distance_matrix[: len(phenotypes), : len(phenotypes)], dtype=bool)
    )
    distance_matrix[
        len(phenotypes) : 2 * len(phenotypes), len(phenotypes) : 2 * len(phenotypes)
    ] = calculate_distance_matrix(go_dense_importances[1], metric=metric)
    distance_matrix_mask[
        len(phenotypes) : 2 * len(phenotypes), len(phenotypes) : 2 * len(phenotypes)
    ] = np.triu(np.ones_like(distance_matrix[: len(phenotypes), : len(phenotypes)], dtype=bool))

    go_dense_importances_0_filt = np.array(go_dense_importances[0])[
        :,
        np.isin(
            graph_csv_dense_index_list[0],
            np.intersect1d(graph_csv_dense_index_list[0], graph_csv_dense_index_list[1]),
        ),
    ]
    go_dense_importances_1_filt = np.array(go_dense_importances[1])[
        :,
        np.isin(
            graph_csv_dense_index_list[1],
            np.intersect1d(graph_csv_dense_index_list[1], graph_csv_dense_index_list[0]),
        ),
    ]
    distance_matrix[len(phenotypes) : 2 * len(phenotypes), : len(phenotypes)] = (
        calculate_distance_matrix_combinations(
            go_dense_importances_0_filt, go_dense_importances_1_filt, metric=metric
        )
    )
    distance_matrix_mask[len(phenotypes) : 2 * len(phenotypes), : len(phenotypes)] = np.triu(
        np.ones_like(distance_matrix[: len(phenotypes), : len(phenotypes)], dtype=bool), k=1
    )

    if mode == "all":
        plot_heatmap_go_graph_similarity(
            distance_matrix,
            distance_matrix_mask,
            phenotypes_labels + phenotypes_labels[:-1] + [""],
            [""] + phenotypes_labels[1:] + phenotypes_labels,
            [len(phenotypes), len(phenotypes)],
            vmin=vmin,
            figsize=(7, 5.8),
            metric=metric,
            save_name=f"neurips2021_cite_bmmc_go_graph_dense_{mode}_{metric}",
        )
    else:
        plot_heatmap_go_graph_similarity(
            distance_matrix,
            distance_matrix_mask,
            phenotypes_labels + phenotypes_labels[:-1] + [""],
            [""] + phenotypes_labels[1:] + phenotypes_labels,
            [len(phenotypes)],
            vmin=vmin,
            figsize=(7, 6.4),
            metric=metric,
            save_name=f"neurips2021_cite_bmmc_go_graph_dense_{mode}_{metric}",
        )


In [ ]:
for metric, vmin in tqdm(zip(["spearman"], vmin_list)):
    distance_matrix = np.zeros((2 * len(phenotypes_cd8_tcells), 2 * len(phenotypes_cd8_tcells)))
    distance_matrix_mask = np.zeros(
        (2 * len(phenotypes_cd8_tcells), 2 * len(phenotypes_cd8_tcells))
    )
    distance_matrix_mask[
        : len(phenotypes_cd8_tcells), len(phenotypes_cd8_tcells) : 2 * len(phenotypes_cd8_tcells)
    ] = 1

    distance_matrix[: len(phenotypes_cd8_tcells), : len(phenotypes_cd8_tcells)] = (
        calculate_distance_matrix(go_cd8_tcells_importances[0], metric=metric)
    )
    distance_matrix_mask[: len(phenotypes_cd8_tcells), : len(phenotypes_cd8_tcells)] = np.triu(
        np.ones_like(
            distance_matrix[: len(phenotypes_cd8_tcells), : len(phenotypes_cd8_tcells)], dtype=bool
        )
    )
    distance_matrix[
        len(phenotypes_cd8_tcells) : 2 * len(phenotypes_cd8_tcells),
        len(phenotypes_cd8_tcells) : 2 * len(phenotypes_cd8_tcells),
    ] = calculate_distance_matrix(go_cd8_tcells_importances[1], metric=metric)
    distance_matrix_mask[
        len(phenotypes_cd8_tcells) : 2 * len(phenotypes_cd8_tcells),
        len(phenotypes_cd8_tcells) : 2 * len(phenotypes_cd8_tcells),
    ] = np.triu(
        np.ones_like(
            distance_matrix[: len(phenotypes_cd8_tcells), : len(phenotypes_cd8_tcells)], dtype=bool
        )
    )

    go_cd8_tcells_importances_0_filt = np.array(go_cd8_tcells_importances[0])[
        :,
        np.isin(
            graph_csv_cd8_tcells_index_list[0],
            np.intersect1d(graph_csv_cd8_tcells_index_list[0], graph_csv_cd8_tcells_index_list[1]),
        ),
    ]
    go_cd8_tcells_importances_1_filt = np.array(go_cd8_tcells_importances[1])[
        :,
        np.isin(
            graph_csv_cd8_tcells_index_list[1],
            np.intersect1d(graph_csv_cd8_tcells_index_list[1], graph_csv_cd8_tcells_index_list[0]),
        ),
    ]
    distance_matrix[
        len(phenotypes_cd8_tcells) : 2 * len(phenotypes_cd8_tcells), : len(phenotypes_cd8_tcells)
    ] = calculate_distance_matrix_combinations(
        go_cd8_tcells_importances_0_filt, go_cd8_tcells_importances_1_filt, metric=metric
    )
    distance_matrix_mask[
        len(phenotypes_cd8_tcells) : 2 * len(phenotypes_cd8_tcells), : len(phenotypes_cd8_tcells)
    ] = np.triu(
        np.ones_like(
            distance_matrix[: len(phenotypes_cd8_tcells), : len(phenotypes_cd8_tcells)], dtype=bool
        ),
        k=1,
    )

    plot_heatmap_go_graph_similarity(
        distance_matrix,
        distance_matrix_mask,
        phenotypes_cd8_tcells_labels + phenotypes_cd8_tcells_labels[:-1] + [""],
        [""] + phenotypes_cd8_tcells_labels[1:] + phenotypes_cd8_tcells_labels,
        [len(phenotypes_cd8_tcells), len(phenotypes_cd8_tcells)],
        vmin=vmin,
        figsize=(3.5, 3),
        metric=metric,
        save_name=f"neurips2021_cite_bmmc_cd8_go_graph_rotated_{mode}_{metric}",
    )


In [ ]:
for metric, vmin in tqdm(zip(["spearman"], vmin_list)):
    distance_matrix = np.zeros((2 * len(phenotypes_cd8_tcells), 2 * len(phenotypes_cd8_tcells)))
    distance_matrix_mask = np.zeros(
        (2 * len(phenotypes_cd8_tcells), 2 * len(phenotypes_cd8_tcells))
    )
    distance_matrix_mask[
        : len(phenotypes_cd8_tcells), len(phenotypes_cd8_tcells) : 2 * len(phenotypes_cd8_tcells)
    ] = 1

    distance_matrix[: len(phenotypes_cd8_tcells), : len(phenotypes_cd8_tcells)] = (
        calculate_distance_matrix(go_cd8_tcells_importances[0], metric=metric)
    )
    distance_matrix_mask[: len(phenotypes_cd8_tcells), : len(phenotypes_cd8_tcells)] = np.triu(
        np.ones_like(
            distance_matrix[: len(phenotypes_cd8_tcells), : len(phenotypes_cd8_tcells)], dtype=bool
        )
    )
    distance_matrix[
        len(phenotypes_cd8_tcells) : 2 * len(phenotypes_cd8_tcells),
        len(phenotypes_cd8_tcells) : 2 * len(phenotypes_cd8_tcells),
    ] = calculate_distance_matrix(go_cd8_tcells_importances[1], metric=metric)
    distance_matrix_mask[
        len(phenotypes_cd8_tcells) : 2 * len(phenotypes_cd8_tcells),
        len(phenotypes_cd8_tcells) : 2 * len(phenotypes_cd8_tcells),
    ] = np.triu(
        np.ones_like(
            distance_matrix[: len(phenotypes_cd8_tcells), : len(phenotypes_cd8_tcells)], dtype=bool
        )
    )

    go_cd8_tcells_importances_0_filt = np.array(go_cd8_tcells_importances[0])[
        :,
        np.isin(
            graph_csv_cd8_tcells_index_list[0],
            np.intersect1d(graph_csv_cd8_tcells_index_list[0], graph_csv_cd8_tcells_index_list[1]),
        ),
    ]
    go_cd8_tcells_importances_1_filt = np.array(go_cd8_tcells_importances[1])[
        :,
        np.isin(
            graph_csv_cd8_tcells_index_list[1],
            np.intersect1d(graph_csv_cd8_tcells_index_list[1], graph_csv_cd8_tcells_index_list[0]),
        ),
    ]
    distance_matrix[
        len(phenotypes_cd8_tcells) : 2 * len(phenotypes_cd8_tcells), : len(phenotypes_cd8_tcells)
    ] = calculate_distance_matrix_combinations(
        go_cd8_tcells_importances_0_filt, go_cd8_tcells_importances_1_filt, metric=metric
    )
    distance_matrix_mask[
        len(phenotypes_cd8_tcells) : 2 * len(phenotypes_cd8_tcells), : len(phenotypes_cd8_tcells)
    ] = np.triu(
        np.ones_like(
            distance_matrix[: len(phenotypes_cd8_tcells), : len(phenotypes_cd8_tcells)], dtype=bool
        ),
        k=1,
    )

    plot_heatmap_go_graph_similarity(
        np.rot90(distance_matrix),
        np.rot90(distance_matrix_mask),
        [""] + phenotypes_cd8_tcells_labels[1:] + phenotypes_cd8_tcells_labels,
        [""] + phenotypes_cd8_tcells_labels[1:][::-1] + phenotypes_cd8_tcells_labels[::-1],
        [len(phenotypes_cd8_tcells), len(phenotypes_cd8_tcells)],
        vmin=vmin,
        figsize=(3, 3),
        metric=metric,
        mirrored_right=True,
        save_name=f"neurips2021_cite_bmmc_cd8_go_graph_rotated_{mode}_{metric}",
    )


# Pathways, GO Groups

In [ ]:
from itertools import combinations


In [ ]:
indices_phenotypes_cd8_tcells = [
    phenotypes.index(phenotype_cd8_tcell) for phenotype_cd8_tcell in phenotypes_cd8_tcells
]


## Expression

In [ ]:
for index12_phenotypes_cd8_tcells in list(combinations(indices_phenotypes_cd8_tcells, 2)):

    go_importances_0 = go_importances[0][index12_phenotypes_cd8_tcells[0]]
    go_importances_1 = go_importances[0][index12_phenotypes_cd8_tcells[1]]

    print(phenotypes_labels[index12_phenotypes_cd8_tcells[0]])
    print(phenotypes_labels[index12_phenotypes_cd8_tcells[1]])

    print("equal ones")
    selector = (go_importances_0 > 0.6) & (go_importances_1 > 0.6)
    print(
        [
            (
                f"{obodag[go_term].name} ({go_term, obodag[go_term].level})"
                if go_term in obodag.keys()
                else go_term
            )
            for go_term in [
                x
                for _, x in sorted(
                    zip(
                        np.abs(go_importances_0 - go_importances_1)[selector],
                        np.array(graph_csv_list[0][0].index)[selector],
                    )
                )
            ][:50]
        ]
    )

    print("diverging")
    print(
        [
            (
                f"{obodag[go_term].name} ({go_term}, {obodag[go_term].level})"
                if go_term in obodag.keys()
                else go_term
            )
            for go_term in [
                x
                for _, x in sorted(
                    zip(np.abs(go_importances_0 - go_importances_1), graph_csv_list[0][0].index)
                )
            ][::-1][:50]
        ]
    )


In [ ]:
for index12_phenotypes_cd8_tcells in list(combinations(list(range(len(phenotypes_cd8_tcells))), 2)):

    go_importances_0 = go_cd8_tcells_importances[0][index12_phenotypes_cd8_tcells[0]]
    go_importances_1 = go_cd8_tcells_importances[0][index12_phenotypes_cd8_tcells[1]]

    print(phenotypes_cd8_tcells_labels[index12_phenotypes_cd8_tcells[0]])
    print(phenotypes_cd8_tcells_labels[index12_phenotypes_cd8_tcells[1]])

    print("equal ones")
    selector = (go_importances_0 > 0.6) & (go_importances_1 > 0.6)
    print(
        [
            (
                f"{obodag[go_term].name} ({go_term, obodag[go_term].level})"
                if go_term in obodag.keys()
                else go_term
            )
            for go_term in [
                x
                for _, x in sorted(
                    zip(
                        np.abs(go_importances_0 - go_importances_1)[selector],
                        np.array(graph_csv_list[0][0].index)[selector],
                    )
                )
            ][:50]
        ]
    )

    print("diverging")
    print(
        [
            (
                f"{obodag[go_term].name} ({go_term}, {obodag[go_term].level})"
                if go_term in obodag.keys()
                else go_term
            )
            for go_term in [
                x
                for _, x in sorted(
                    zip(np.abs(go_importances_0 - go_importances_1), graph_csv_list[0][0].index)
                )
            ][::-1][:50]
        ]
    )


## Expression differences overall/ subgroup

In [ ]:
for index12_phenotypes_full_cd8_tcells, index12_phenotypes_separate_cd8_tcells in zip(
    list(combinations(indices_phenotypes_cd8_tcells, 2)),
    list(combinations(list(range(len(phenotypes_cd8_tcells))), 2)),
):

    go_importances_full_cd8_tcells_0 = go_importances[0][index12_phenotypes_full_cd8_tcells[0]]
    go_importances_full_cd8_tcells_1 = go_importances[0][index12_phenotypes_full_cd8_tcells[1]]

    go_importances_separate_cd8_t_cells_0 = go_cd8_tcells_importances[0][
        index12_phenotypes_separate_cd8_tcells[0]
    ]
    go_importances_separate_cd8_t_cells_1 = go_cd8_tcells_importances[0][
        index12_phenotypes_separate_cd8_tcells[1]
    ]

    selector = (
        (go_importances_full_cd8_tcells_0 > 0.6)
        & (go_importances_full_cd8_tcells_1 > 0.6)
        & (go_importances_separate_cd8_t_cells_0 > 0.6)
        & (go_importances_separate_cd8_t_cells_1 > 0.6)
    )

    print(phenotypes_cd8_tcells_labels[index12_phenotypes_separate_cd8_tcells[0]])
    print(phenotypes_cd8_tcells_labels[index12_phenotypes_separate_cd8_tcells[1]])

    diverging_go_importances_full_cd8_tcells = np.abs(
        go_importances_full_cd8_tcells_0[selector] - go_importances_full_cd8_tcells_1[selector]
    )
    diverging_go_importances_separate_cd8_tcells = np.abs(
        go_importances_separate_cd8_t_cells_0[selector]
        - go_importances_separate_cd8_t_cells_1[selector]
    )
    diverging_go_importances_full_cd8_tcells = np.abs(
        diverging_go_importances_full_cd8_tcells - diverging_go_importances_separate_cd8_tcells
    )
    print(
        [
            (
                f"{obodag[go_term].name} ({go_term}, {obodag[go_term].level})"
                if go_term in obodag.keys()
                else go_term
            )
            for go_term in [
                x
                for _, x in sorted(
                    zip(
                        diverging_go_importances_full_cd8_tcells,
                        graph_csv_list[0][0].index[selector],
                    )
                )
            ][::-1][:50]
        ]
    )


## Protein

In [ ]:
for index12_phenotypes_cd8_tcells in list(combinations(indices_phenotypes_cd8_tcells, 2)):

    go_importances_0 = go_importances[1][index12_phenotypes_cd8_tcells[0]]
    go_importances_1 = go_importances[1][index12_phenotypes_cd8_tcells[1]]

    print(phenotypes_labels[index12_phenotypes_cd8_tcells[0]])
    print(phenotypes_labels[index12_phenotypes_cd8_tcells[1]])

    print("equal ones")
    selector = (go_importances_0 > 0.6) & (go_importances_1 > 0.6)
    print(
        [
            (
                f"{obodag[go_term].name} ({go_term, obodag[go_term].level})"
                if go_term in obodag.keys()
                else go_term
            )
            for go_term in [
                x
                for _, x in sorted(
                    zip(
                        np.abs(go_importances_0 - go_importances_1)[selector],
                        np.array(graph_csv_list[1][0].index)[selector],
                    )
                )
            ][:50]
        ]
    )

    print("diverging")
    print(
        [
            (
                f"{obodag[go_term].name} ({go_term}, {obodag[go_term].level})"
                if go_term in obodag.keys()
                else go_term
            )
            for go_term in [
                x
                for _, x in sorted(
                    zip(np.abs(go_importances_0 - go_importances_1), graph_csv_list[1][0].index)
                )
            ][::-1][:50]
        ]
    )


In [ ]:
for index12_phenotypes_cd8_tcells in list(combinations(list(range(len(phenotypes_cd8_tcells))), 2)):

    go_importances_0 = go_cd8_tcells_importances[1][index12_phenotypes_cd8_tcells[0]]
    go_importances_1 = go_cd8_tcells_importances[1][index12_phenotypes_cd8_tcells[1]]

    print(phenotypes_cd8_tcells_labels[index12_phenotypes_cd8_tcells[0]])
    print(phenotypes_cd8_tcells_labels[index12_phenotypes_cd8_tcells[1]])

    print("equal ones")
    selector = (go_importances_0 > 0.6) & (go_importances_1 > 0.6)
    print(
        [
            (
                f"{obodag[go_term].name} ({go_term, obodag[go_term].level})"
                if go_term in obodag.keys()
                else go_term
            )
            for go_term in [
                x
                for _, x in sorted(
                    zip(
                        np.abs(go_importances_0 - go_importances_1)[selector],
                        np.array(graph_csv_list[1][0].index)[selector],
                    )
                )
            ][:50]
        ]
    )

    print("diverging")
    print(
        [
            (
                f"{obodag[go_term].name} ({go_term}, {obodag[go_term].level})"
                if go_term in obodag.keys()
                else go_term
            )
            for go_term in [
                x
                for _, x in sorted(
                    zip(np.abs(go_importances_0 - go_importances_1), graph_csv_list[1][0].index)
                )
            ][::-1][:50]
        ]
    )


## Protein differences overall/ subgroup

In [ ]:
for index12_phenotypes_full_cd8_tcells, index12_phenotypes_separate_cd8_tcells in zip(
    list(combinations(indices_phenotypes_cd8_tcells, 2)),
    list(combinations(list(range(len(phenotypes_cd8_tcells))), 2)),
):

    go_importances_full_cd8_tcells_0 = go_importances[1][index12_phenotypes_full_cd8_tcells[0]]
    go_importances_full_cd8_tcells_1 = go_importances[1][index12_phenotypes_full_cd8_tcells[1]]

    go_importances_separate_cd8_t_cells_0 = go_cd8_tcells_importances[1][
        index12_phenotypes_separate_cd8_tcells[0]
    ]
    go_importances_separate_cd8_t_cells_1 = go_cd8_tcells_importances[1][
        index12_phenotypes_separate_cd8_tcells[1]
    ]

    selector = (
        (go_importances_full_cd8_tcells_0 > 0.6)
        & (go_importances_full_cd8_tcells_1 > 0.6)
        & (go_importances_separate_cd8_t_cells_0 > 0.6)
        & (go_importances_separate_cd8_t_cells_1 > 0.6)
    )

    print(phenotypes_cd8_tcells_labels[index12_phenotypes_separate_cd8_tcells[0]])
    print(phenotypes_cd8_tcells_labels[index12_phenotypes_separate_cd8_tcells[1]])

    diverging_go_importances_full_cd8_tcells = np.abs(
        go_importances_full_cd8_tcells_0[selector] - go_importances_full_cd8_tcells_1[selector]
    )
    diverging_go_importances_separate_cd8_tcells = np.abs(
        go_importances_separate_cd8_t_cells_0[selector]
        - go_importances_separate_cd8_t_cells_1[selector]
    )
    diverging_go_importances_full_cd8_tcells = np.abs(
        diverging_go_importances_full_cd8_tcells - diverging_go_importances_separate_cd8_tcells
    )
    print(
        [
            (
                f"{obodag[go_term].name} ({go_term}, {obodag[go_term].level})"
                if go_term in obodag.keys()
                else go_term
            )
            for go_term in [
                x
                for _, x in sorted(
                    zip(
                        diverging_go_importances_full_cd8_tcells,
                        graph_csv_list[1][0].index[selector],
                    )
                )
            ][::-1][:50]
        ]
    )


## Fig. 2e, S17a-d: marker-gene dropout sensitivity analysis (CD19, CD4, CD8A, CD38)

In [ ]:
from goatools.anno.gaf_reader import GafReader

ensembl2go = []
map_ensembl_list = [
    os.path.join(os.getcwd(), "..", "preprocessing", "resources", "ebi_ensembl_go_mappings", "goa_human_ensembl_gene_mapping_2024-08-08.gaf"),
    os.path.join(os.getcwd(), "..", "preprocessing", "resources", "ebi_ensembl_go_mappings", "goa_human_isoform_ensembl_gene_mapping_2024-08-08.gaf"),
    os.path.join(os.getcwd(), "..", "preprocessing", "resources", "ebi_ensembl_go_mappings", "goa_human_rna_ensembl_gene_mapping_2024-08-08.gaf"),
]
for map_ensembl in map_ensembl_list:
    objanno = GafReader(map_ensembl)
    ensembl2go.append(objanno.get_ns2assc()["BP"])


In [ ]:
import textwrap

def wrap_labels(labels, max_characters=10):
    return ["\n".join(textwrap.wrap(label, max_characters, break_long_words=False)) for label in labels]

def plot_perturbation_plot(mode, selected_perturbation_go, selected_perturbation_go_ids, selected_go_importances_perturbation, phenotypes_labels, go_terms_perturbation_gene, go_term_parents_perturbation_gene, figsize=(10, 20), bold_go_terms=None, save_name=False):
    selected_perturbation_go_wrapped = wrap_labels(selected_perturbation_go, max_characters=70)

    data = np.array(selected_go_importances_perturbation).T
    vmin = np.min(data)
    vmax = np.max(data)
    midpoint = 0
    norm = TwoSlopeNorm(vmin=vmin, vcenter=midpoint, vmax=vmax)

    if mode == "all":
        clustered_heatmap = sns.clustermap(data, norm=norm, figsize=(3.25, 2.8), xticklabels=phenotypes_labels, yticklabels=selected_perturbation_go_wrapped, cmap=cmap_heat, row_cluster=True, col_cluster=False)
    else:
        clustered_heatmap = sns.clustermap(data, norm=norm, figsize=(3.25, 2.6), xticklabels=phenotypes_labels, yticklabels=selected_perturbation_go_wrapped, cmap=cmap_heat, row_cluster=True, col_cluster=False)

    clustered_heatmap.ax_row_dendrogram.set_visible(False)
    ax = clustered_heatmap.ax_heatmap
    colorbar = clustered_heatmap.cax

    if mode == "all":
        colorbar.set_position([1.00, 0.6, 0.02, 0.2])
    else:
        colorbar.set_position([1.01, 0.6, 0.02, 0.2])

    yticks = ax.get_yticklabels()
    ax.set_yticklabels(yticks)

    if mode == "subset":
        bold_labels = [
            "Lymph prog", "Naive CD20+ B IGKC+", "Naive CD20+ B IGKC-",
            "Plasma cell IGKC+", "Plasma cell IGKC-", "Plasmablast IGKC+", "Plasmablast IGKC-",
        ]
        xticks = ax.get_xticklabels()
        for tick in xticks:
            if tick.get_text() in bold_labels:
                tick.set_fontweight("bold")

    if bold_go_terms is not None:
        for tick in ax.get_yticklabels():
            if tick.get_text() in bold_go_terms:
                tick.set_fontweight("bold")

    ax.grid(False)
    plt.tight_layout()

    if save_name:
        plt.savefig(os.path.join(_OUTDIR_ABS, f"{save_name}.png"), dpi=500, format="png", transparent=True, bbox_inches="tight")
        plt.savefig(os.path.join(_OUTDIR_ABS, f"{save_name}.svg"), dpi=500, transparent=True, bbox_inches="tight", format="svg")
    else:
        plt.show()


In [ ]:
def _run_perturbation_analysis(perturbation_genes):
    mapping_modality = {"expression": "Gene Expression_all", "accessibility": "Peaks_all", "protein": "protein"}

    for perturbation_gene in perturbation_genes:
        graph_csv_list = []
        graph_csv_index_list = []
        go_importances_perturbation = []

        for modality in ["protein"]:
            go_importances_modality_perturbation = []
            graph_csv_modality_list = []
            for phenotype in phenotypes:
                graph_csv = pd.read_pickle(
                    f"../.experiment_data/scratch/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/calc_go_terms_cell_type/{modality}_cell_type_vs_rest_{phenotype}_res_unp_act_mask_False_flip_False_go_terms_evaluated_graph_csv.pkl"
                )
                perturbation_path = f"../.experiment_data/scratch/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/calc_go_terms_cell_type/cell_type_go_terms_perturbation_{perturbation_gene}_evaluated_graph_csv.pkl"
                with open(perturbation_path, "rb") as f:
                    perturbation = pickle.load(f)

                graph_csv["predictors_label_valid_roc"] = graph_csv["predictors_label_valid_roc"].astype("float")
                graph_csv["predictors_label_valid_roc_perturbation"] = (
                    graph_csv["predictors_label_valid_roc"] - perturbation[mapping_modality[modality]][modality][phenotype]
                ).astype("float")
                graph_csv_modality_list.append(graph_csv)
                go_importances_modality_perturbation.append(np.array(graph_csv["predictors_label_valid_roc_perturbation"]))

            graph_csv_list.append(graph_csv_modality_list)
            graph_csv_index_list.append(np.array(graph_csv_modality_list[0].index))
            go_importances_perturbation.append(go_importances_modality_perturbation)

        go_terms_perturbation_gene = set(
            go_term
            for ensembl2go_sub in ensembl2go
            if perturbation_gene in ensembl2go_sub.keys()
            for go_term in ensembl2go_sub[perturbation_gene]
        )
        go_term_parents_perturbation_gene = set()
        for go_term_perturbation_gene in go_terms_perturbation_gene:
            go_term_parents_perturbation_gene = go_term_parents_perturbation_gene | obodag[go_term_perturbation_gene].get_all_parents()
        go_term_parents_perturbation_gene = go_term_parents_perturbation_gene - go_terms_perturbation_gene

        for modality_index, (modality, figsize) in enumerate(zip(["protein"], [(14, 8)])):
            filtering_threshold = 0.0015 if modality == "protein" else 0.005

            go_indices = set()
            for phenotype_label_index, phenotype_label in enumerate(phenotypes_labels):
                vals = go_importances_perturbation[modality_index][phenotype_label_index]
                downregulated = set(np.where(vals < np.quantile(vals, filtering_threshold))[0])
                upregulated = set(np.where(vals > np.quantile(vals, 1 - filtering_threshold))[0])
                go_indices = go_indices | downregulated | upregulated
            go_indices = list(go_indices)
            figsize = (figsize[0], int(len(go_indices) * 0.33))

            selected_go_importances_perturbation = [
                go_importances_perturbation[modality_index][phenotype_label_index][go_indices]
                for phenotype_label_index in range(len(phenotypes_labels))
            ]

            go_label_replacement = {
                "regulation of sequestering of calcium ion": "reg. sequestering of calcium ion",
                "positive regulation of developmental process": "pos. reg. of developmental process",
                "biological process involved in interaction with host": "biol. process inv. in int. with host",
                "regulation of biological process involved in symbiotic interaction": "reg. of biol. proc. inv. in symbiotic int.",
                "somatic diversification of immunoglobulins involved in immune response": "som. div. immunoglob. inv. in immune resp.",
                "cell activation involved in immune response": "cell act. in immune response",
                "leukocyte activation involved in immune response": "leukocyte act. in immune response",
                "positive regulation of immune system process": "pos. reg. of immune system process",
                "adaptive immune response based on somatic recombination of immune receptors built from immunoglobulin superfamily domains": "imm. resp. via som. recomb. of rec.",
            }
            selected_perturbation_go = [
                f"{go_label_replacement.get(obodag[go_term].name, obodag[go_term].name)} "
                for go_term in graph_csv_index_list[modality_index][go_indices]
            ]
            selected_perturbation_go_ids = [
                f"{go_label_replacement.get(obodag[go_term].name, obodag[go_term].name)} ({go_term})"
                for go_term in graph_csv_index_list[modality_index][go_indices]
            ]

            bold_go_terms = [
                go_label_replacement.get(obodag[go_term].name, obodag[go_term].name)
                for go_term in (go_term_parents_perturbation_gene | go_terms_perturbation_gene)
            ]

            plot_perturbation_plot(
                mode,
                selected_perturbation_go,
                selected_perturbation_go_ids,
                selected_go_importances_perturbation,
                phenotypes_labels,
                go_terms_perturbation_gene,
                go_term_parents_perturbation_gene,
                figsize,
                bold_go_terms=bold_go_terms,
                save_name=f"neurips2021_cite_bmmc_perturbation_{mode}_{modality}_{perturbation_gene}",
            )

_run_perturbation_analysis(["ENSG00000010610", "ENSG00000153563", "ENSG00000004468"])


In [ ]:
mode = "subset"
phenotypes = [
    "Lymph_prog", "G_M_prog", "MK_E_prog", "CD4+_T_naive", "CD8+_T_naive",
    "Naive_CD20+_B_IGKC+", "Naive_CD20+_B_IGKC-",
    "Plasma_cell_IGKC+", "Plasma_cell_IGKC-", "Plasmablast_IGKC+", "Plasmablast_IGKC-",
    "pDC", "Erythroblast", "Normoblast", "Reticulocyte",
]
phenotypes_labels = [
    "Lymph prog", "G/M prog", "MK/E prog", "CD4+ T naive", "CD8+ T naive",
    "Naive CD20+ B IGKC+", "Naive CD20+ B IGKC-",
    "Plasma cell IGKC+", "Plasma cell IGKC-", "Plasmablast IGKC+", "Plasmablast IGKC-",
    "pDC", "Erythroblast", "Normoblast", "Reticulocyte",
]

_run_perturbation_analysis(["ENSG00000177455"])

mode = "all"


## S17e: IRF8 sensitivity analysis (pDC-specific)

In [ ]:
_irf8_base_dir = (
    "../.experiment_data/primary/experiments_new/"
    "sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_"
    "scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_"
    "ENCFF173VDJ_gaf_standard_with_rna_s0/default/calc_go_terms_cell_type"
)
_irf8_gene = "ENSG00000140968"
_irf8_modality = "expression"

mode = "all"
phenotypes = ["CD8+_T_naive", "CD16+_Mono", "CD4+_T_naive", "CD14+_Mono", "NK", "pDC", "Naive_CD20+_B_IGKC+"]
phenotypes_labels = ["CD8+ T naive", "CD16+ Mono", "CD4+ T naive", "CD14+ Mono", "NK", "pDC", "Naive CD20+ B IGKC+"]

graph_csv_modality_list = []
for phenotype in phenotypes:
    graph_csv = pd.read_pickle(
        os.path.join(
            _irf8_base_dir,
            f"{_irf8_modality}_cell_type_vs_rest_{phenotype}_res_unp_act_mask_False_flip_False_go_terms_evaluated_graph_csv.pkl",
        )
    )
    graph_csv["predictors_label_valid_roc"] = graph_csv["predictors_label_valid_roc"].astype("float")

    perturbation_path = os.path.join(
        _irf8_base_dir,
        f"cell_type_vs_rest_res_unp_act_mask_False_flip_False_go_terms_perturbation_{_irf8_gene}_zero_evaluated_graph_csv_with_ci.pkl",
    )
    with open(perturbation_path, "rb") as f:
        perturbation = pickle.load(f)

    phenotype_label = phenotypes_labels[phenotypes.index(phenotype)]
    graph_csv["predictors_label_valid_roc_perturbation"] = (
        graph_csv["predictors_label_valid_roc"] - perturbation[_irf8_modality][_irf8_modality][phenotype_label]["roc"]
    ).astype("float")
    graph_csv_modality_list.append(graph_csv)

graph_csv_index = np.array(graph_csv_modality_list[0].index)
go_importances_perturbation = [np.array(g["predictors_label_valid_roc_perturbation"]) for g in graph_csv_modality_list]

parents_to_exclude = {"GO:0032502", "GO:0008152", "GO:0008283", "GO:0048870"}
keep_mask = np.array([
    (go_term not in parents_to_exclude)
    and (go_term not in obodag.keys() or not any(p in parents_to_exclude for p in obodag[go_term].get_all_parents()))
    for go_term in graph_csv_index
])
keep_indices = np.where(keep_mask)[0]

filtering_threshold = 0.005
go_indices_local = set()
for phenotype_label_index in range(len(phenotypes_labels)):
    arr = go_importances_perturbation[phenotype_label_index][keep_indices]
    go_indices_local |= set(np.where(arr < np.quantile(arr, filtering_threshold))[0])
    go_indices_local |= set(np.where(arr > np.quantile(arr, 1 - filtering_threshold))[0])
go_indices = keep_indices[list(go_indices_local)]

figsize = (14, int(len(go_indices) * 0.33))

selected_go_importances_perturbation = [
    go_importances_perturbation[phenotype_label_index][go_indices]
    for phenotype_label_index in range(len(phenotypes_labels))
]

go_terms_perturbation_gene = set(
    go_term
    for ensembl2go_sub in ensembl2go
    if _irf8_gene in ensembl2go_sub.keys()
    for go_term in ensembl2go_sub[_irf8_gene]
)
go_term_parents_perturbation_gene = set()
for gt in go_terms_perturbation_gene:
    go_term_parents_perturbation_gene |= obodag[gt].get_all_parents()
go_term_parents_perturbation_gene -= go_terms_perturbation_gene

selected_perturbation_go = [f"{obodag[go_term].name} " for go_term in graph_csv_index[go_indices]]
selected_perturbation_go_ids = [f"{obodag[go_term].name} ({go_term})" for go_term in graph_csv_index[go_indices]]

plot_perturbation_plot(
    mode,
    selected_perturbation_go,
    selected_perturbation_go_ids,
    selected_go_importances_perturbation,
    phenotypes_labels,
    go_terms_perturbation_gene,
    go_term_parents_perturbation_gene,
    figsize,
    save_name=f"neurips2021_cite_bmmc_perturbation_{mode}_{_irf8_modality}_{_irf8_gene}",
)

mode = "all"


## pDC activation UMAP + ranked-quantile IRF8/BCL11A dotplot

In [ ]:
def show_umap_plot(X_umap, labels, cmap, continuous=False, save_name=False, vmin=None, vmax=None):
    set_figure_params(4, 4)
    fig, ax = plt.subplots()
    ax.set_aspect("equal", "box")
    sc_ax = ax.scatter(X_umap[:, 0], X_umap[:, 1], s=0.3, c=labels, cmap=cmap, edgecolor="none", vmin=vmin, vmax=vmax)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_box_aspect(1)
    if save_name:
        plt.savefig(os.path.join(_OUTDIR_ABS, f"{save_name}.png"), dpi=500, format="png", transparent=True, bbox_inches="tight")
        plt.savefig(os.path.join(_OUTDIR_ABS, f"{save_name}.svg"), dpi=500, transparent=True, bbox_inches="tight", format="svg")
    else:
        plt.show()
    plt.close()

_X_umap_full = np.load(
    "../.experiment_data/primary/experiments/"
    "sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_"
    "batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/X_umap.npy"
)
_go_term_pdc = "GO:0002270"
_pdc_graph_csv = pd.read_pickle(
    "../.experiment_data/scratch/experiments/"
    "sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_"
    "batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/"
    "calc_go_terms_cell_type/expression_cell_type_vs_rest_pDC_res_unp_act_mask_False_flip_False_go_terms_evaluated_graph_csv.pkl"
)
# Raw per-cell activation is heavily left-skewed (long negative outlier tail among a
# minority of cells); -mean flips sign so the pDC-high cells become the darkest extreme.
# A small number of extreme outliers otherwise blow out the color scale and wash out
# everything else to near-white, so the top end is capped at the 90th percentile
# (matches the higher-contrast version copied directly from the notebook's inline render).
_go_activations_full_raw = -np.mean(_pdc_graph_csv.loc[_go_term_pdc]["activations_valid"], axis=1)
_go_activations_vmax = np.percentile(_go_activations_full_raw, 90)

show_umap_plot(
    _X_umap_full,
    _go_activations_full_raw,
    cmap,
    continuous=True,
    save_name="neurips2021_cite_bmmc_pdc_activation_umap",
    vmin=_go_activations_full_raw.min(),
    vmax=_go_activations_vmax,
)


**Note:** dotplot below is reconstructed, not an original figure cell.

In [ ]:
import scanpy as sc
import anndata as ad

_resorted_mdata_counts = mdata_counts.mod["rna"][train_indices].copy()
_indices_pdc = np.where(_resorted_mdata_counts.obs[labels_key] == "pDC")[0]

_pdc_activations_2d = _pdc_graph_csv.loc[_go_term_pdc]["activations_valid"][_indices_pdc]
_pdc_activation_score = np.mean(_pdc_activations_2d, axis=1)

_pdc_adata = _resorted_mdata_counts[_indices_pdc, ["IRF8", "BCL11A"]].copy()
_quantile_labels = ["0-20%", "20-40%", "40-60%", "60-80%", "80-100%"]
_pdc_adata.obs["pDC activation quantile"] = pd.qcut(_pdc_activation_score, 5, labels=_quantile_labels)

_pdc_adata_all = _pdc_adata.copy()
_pdc_adata_all.obs["pDC activation quantile"] = "All pDC"
_pdc_adata_combined = ad.concat([_pdc_adata, _pdc_adata_all])
_pdc_adata_combined.obs["pDC activation quantile"] = pd.Categorical(
    _pdc_adata_combined.obs["pDC activation quantile"], categories=_quantile_labels + ["All pDC"]
)

_dp = sc.pl.dotplot(
    _pdc_adata_combined,
    ["IRF8", "BCL11A"],
    groupby="pDC activation quantile",
    categories_order=_quantile_labels + ["All pDC"],
    show=False,
    return_fig=True,
)
_dp.savefig(
    os.path.join(_OUTDIR_ABS, "neurips2021_cite_bmmc_pdc_activation_ranked_IRF8_BCL11A_dotplot.png"),
    dpi=500, format="png", transparent=True, bbox_inches="tight",
)
_dp.savefig(
    os.path.join(_OUTDIR_ABS, "neurips2021_cite_bmmc_pdc_activation_ranked_IRF8_BCL11A_dotplot.svg"),
    dpi=500, transparent=True, bbox_inches="tight", format="svg",
)
plt.show()
